<table style="width: 100%; border-style: none;">
<tr style="border-style: none">
<td style="border-style: none; width: 1%; text-align: left; font-size: 16px">Institut f&uuml;r Theoretische Physik<br /> Universit&auml;t zu K&ouml;ln</td>
<td style="border-style: none; width: 1%; font-size: 16px">&nbsp;</td>
<td style="border-style: none; width: 1%; text-align: right; font-size: 16px">Prof. Dr. Matteo Rizzi

</tr>
</table>
<hr>
<h1 style="font-weight:bold; text-align: center; margin: 0px; font-size: 30px; padding:0px;">Computerphysik</h1>
<h1 style="font-weight:bold; text-align: center; margin: 0px; font-size: 30px; padding:0px;">Übungsblatt 6</h1>
<hr>
<h3 style="font-weight:bold; text-align: center; margin: 0px; font-size: 20px; padding:0px; margin-bottom: 20px;">Sommersemester 2026</h3>
<hr>

<table style="border-style: none; width: 100%"><tr style="border-style: none;">
<td style="border-style: none; width: 1%; text-align: left; font-size: 25px; font-weight: bold; text-decoration: underline">Aufgaben auf Übungsblatt 7</td>
<td style="border-style: none; width: 1%; text-align: right; font-size: 15px"></td></tr></table>

- Aufgabe 07_01_A
- Aufgabe 07_01_B
- **Aufgabe 07_02**

## Aufgabe 2) Die schwingende Saite

---

In dieser Aufgabe sollen Sie die Wellengleichung einer eindimensionalen schwingenden Saite numerisch untersuchen:

$$
    \partial_t^2 u(t,x) = c^2 \partial_x^2 u(t,x) \, .\quad (1)
$$

Dabei beschreibt $u(t,x)$ die Auslenkung der Saite aus der Ruhelage $u=0$, $\partial_t$ bzw. $\partial_x$ stehen für die partiellen Ableitungen in Zeit bzw. Raum und `c` ist die Ausbreitungsgeschwindigkeit der Welle.

Da dies eine zeitabhängige partielle Differentialgleichung ist, benötigt man sowohl die Anfangsbedingungen

$$
    \begin{matrix}
    u(0,x) &= u_0(x) \quad &(I1)\\ 
    \partial_t u(0,x) &= v_0(x) \quad &(I2)
    \end{matrix}
$$

als auch Randbedingungen, welche in unterschiedlicher Form auftreten können

$$
    \begin{matrix}
        u(t,0) = f_L(t)                            & \land & u(t,L) = f_R(x) \quad                            & \text{(Dirichlet)} \\
        \partial_x u(t,0) = q_L(t)                 & \land & \partial_x u(t,L) = q_R(t) \quad                 & \text{(von Neumann)} \\
        \partial_t u(t,0) - c\partial_x u(t,0) = 0 & \land & \partial_t u(t,L) + c\partial_x u(t,L) = 0 \quad & \text{(open boundary conditions)}
    \end{matrix}
$$

als auch jegliche gemischte Formen.

### a) Diskretisierung der Differentialgleichung

---

Bestimmen Sie mithilfe der Diskretisierung der zweifachen Ableitung:

$$
    \left(\partial^2_x g\right)|_x \rightarrow \frac{g(x+\epsilon) - 2g(x) + g(x-\epsilon)}{\epsilon^2}
$$

eine Iterationsvorschrift

$$
    u^{n+1}_j = f(u^n_j, u^{n-1}_j)
$$

welche die Lösung zur Zeit $t_{n}$ und $t_{n-1}$ in die Lösung zur Zeit $t_{n+1}$ transformiert. Dabei ist $u^n_j := u((n-1)\delta t, (j-1)\delta x)$ und $\delta t$ und $\delta x$ die Diskretisierungsschritte in Zeit und Raum.

Vervollständigen Sie für Dirichlet-Randbedingungen die folgende `timestep`-Funktion, welche das vorherige und jetzige Auslenkprofil ($u_0$, $u_1$) das nächste Auslenkprofil berechnet. Setzen Sie dafür $c = 1$. Die Funktion bekommt außerdem die Diskretisierungsschritte $\delta t$ und $\delta x$ übergeben sowie die Werte an den Rändern $x_1$ und $x_N$ für den aktuellen Zeitschritt.

In [1]:
# Funktion zum updaten von der aktuellen Auslenkung u_1 zum nächsten Schritt. 
# Gibt den aktuellen Schritt und den nächsten Schritt zurueck
function timestep(u_0, u_1, δt, δx, u_left, u_right)
    κ = δt/δx
    N = length(u_1)
    u_next = zeros(N)
    u_next[1] = u_left
    u_next[N] = u_right

    for i in 2:N-1
    u_next[i] = 2(1-κ^2) * u_1[i] + κ^2(u_1[i+1]-u_1[i-1]) - u_0[i]
    end
    
    return u_1, u_next
end

timestep (generic function with 1 method)

---

Nachdem die DGL diskretisiert wurde, müssen auch die Randbedingungen und Anfangsbedingungen noch diskretisiert und implementiert werden.

Während die Anfangsbedingung $(I1)$ und die Dirichlet-Randbedingung (DR) einfach numerisch zu implementieren sind, muss $(I2)$ auch noch diskretisiert werden.
Dafür eignet sich am besten die Vorschrift:
$$
    \partial_t u(t,x) \rightarrow \frac{1}{2\delta t}\left(u^{n+1}_j - u^{n-1}_j \right) 
$$
Für $t=0$ ($n=1$) führt auf eine Bestimmungsgleichung für $u^0_j$, welche nur noch von $u^2_j$ abhängig ist. Da $u^0_j$ einem Zeitpunkt vor unserer Simulation entspricht, kann man diese Bestimmungsgleichung
dann nutzen, um $u^0_j$ aus der Iterationsvorschrift für $u^2_j$ zu entfernen, was dann zu einer modifizierten Form
$u^2_j = \tilde{f}(u^1_j)$ führt, welche nur von der Anfangskonfiguration $u^1_j$ abhängig ist.

Die Angabe von $u^1_j$ und $u^2_j$ zusammen mit den Randbedingungen für alle Zeiten komplettiert die numerische Prozedur.

> **Aufgabe**: 
> - Bestimmen Sie $\tilde{f}$
> - Erstellen Sie eine Initialisierungsfunktion, um $u^1_j$ und $u^2_j$ mit einem vorgegebenen Startprofil und Geschwindigkeitsprofil zu initialisieren.

Setzen Sie dafür $L=1$.

**Hinweis**: 
Die Zahl $\kappa:=c\delta t/\delta x$ wird auch Courant Zahl genannt, und numerische Stabilität ist garantiert für $\kappa \le 1$

In [ ]:
# Funktion, welche aus einer Funktion für die Anfangsform und eine Funktion für die Anfangsgeschwindigkeit 
# u^1_j und u^2_j erzeugt.
function initialize(shape_fn, velocity_profile,δt, δx)
    #=
    Hier sollte Ihr Code stehen!
    u_0: Array, welches u(t=0,x) darstellt
    u_1: Array, welches u(t=δt,x) darstellt
    x: Array, welches die Diskretisierung des Raumes darstellt
    =#
    return u_0,u_1,x
end;

### b) Visualisierung der Lösung

---

Die Funktion `animate` nimmt nun Ihre Startkonfiguration `u_0` und `u_1` zusammen mit der Orts- und Zeitdiskretisierung $\delta x$ und $\delta t$ entgegen, zusammen mit 
der finalen Zeit, bis zu der das System simuliert werden soll. Die Zeitevolution wird dazu mit Ihrer zuvor definierten `timestep`-Funktion durchgeführt.

Optional können Sie noch die Dirichlet-Randbedingungen als Funktion an `animate` übergeben:
```
my_bc(t) = sin(t)
anim = animate(u_0,u_1, δx,δt, T; bc_left = my_bc)
```
würde z. B. am linken Rand eine oszillierende Funktion als Randbedingung besitzen.
Die Animation kann dann durch
```
gif(anim, "ex1.gif", fps = 15)
```
erzeugt werden.


Implementieren Sie nun verschiedene Startprofile und Randbedingungen und spielen Sie ein wenig herum. Mögliche Kombinationen:

$$
    \begin{cases}
        a) & u_0(x) = 0,                       &  v_0(x) = 0,                   & \mathrm{bc}_\mathrm{right}(t) = 0, \quad \mathrm{bc}_\mathrm{left} = A\sin(\omega t)\exp(-\mu t) \\
        b) & u_0(x) = exp(-(x-x0)^2/\sigma^2), & v_0(x) = 0,                    & \mathrm{bc}_\mathrm{right}(t) = \mathrm{bc}_\mathrm{left}(t) = 0 \\
        c) & u_0(x) = 0,                       & v_0 = exp(-(x-x0)^2/\sigma^2), & \mathrm{bc}_\mathrm{right}(t) = \mathrm{bc}_\mathrm{left}(t) = 0
    \end{cases}
$$



In [ ]:
function animate(u_0,u_1, δx,δt, T; bc_left = t->0, bc_right = t ->0)
    default(label = false)
    nt = Int(T/δt) + 1
    #δx = x[2]-x[1]
    x = 0:δx:1
    u_current = copy(u_1)
    u_prev    = copy(u_0)
    
    n_save = 10
    
    anim = @animate for jj in 1:n_save:nt
        if(jj == 1)
            plot(x, u_prev, xlim = (0, 1), ylim = (-1,1), markershape = :circle, color = :black)
        elseif(jj == 2)
            plot(x, u_current,  xlim = (0, 1), ylim = (-1,1), markershape = :circle, color = :black)
        else
            for kk = jj:jj+n_save-1
                u_prev, u_current = timestep(u_prev, u_current, δt, δx, bc_left(kk*δt), bc_right(kk*δt))
            end
            plot(x, u_current, xlim = (0, 1), ylim = (-1,1), markershape = :circle, color = :black)
        end
    end
    
    return anim
end

In [ ]:
#=
Hier sollte Ihr Code stehen!
=#

### c) Bonus Implementierung von OBC

---

Um auch andere Randbedingungen zu implementieren, müssen auch diese diskretisiert werden. In dieser Bonusaufgabe sollen nun die Dirichlet-Randbedingungen auf der rechten Seite durch offene Randbedingungen ersetzt werden.

Die offenen Randbedingungen am rechten Rand werden durch:

$$
    \partial_t u(t,L) = - c\partial_x u(t,L)
$$

bestimmt und verknüpfen die räumlichen mit den zeitlichen Ableitungen.
Fällt Ihnen eine schlaue Möglichkeit ein, die OBC Randbedingungen so zu diskretisieren, dass man eine gesonderte Iterationsvorschrift für $u^{n+1}_L$ erhält? Tipp: Diese darf nur von den vorherigen Zeitschritten abhängen.


Modifizieren Sie damit die `timestep`- und `animation`-Methode um offene Randbedingungen zu simulieren.

In [ ]:
#=
Hier sollte Ihr Code stehen!
=#